# Agents and tool use: worked solution

Worked solution for the exercise in the lesson **[Agents and tool use](https://ramkumarmd.com/medical-ai/agentic-ai/agents-and-tool-use)** (*Agentic AI* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](01-agents-and-tool-use.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** An AI agent is a language model that works in a loop: it decides which tool to call (a calculator, a record search, a guideline lookup), reads the result, and repeats until it can answer or act. Tools let models do things they are bad at on their own, such as exact calculation and looking up current data. They also add new risks: wrong tool inputs, actions with real consequences, and instructions hidden in tool results. Clinical agents need narrow, validated, mostly read-only tools, step limits, full logging and human approval for any action.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Describing tools

Two tools: an eGFR calculator and a protocol lookup.

In [ ]:
import json

# Tools the agent may call, described in the JSON-schema style most model APIs use
TOOLS = {
    "egfr_ckd_epi_2021": {
        "description": "Estimate eGFR (mL/min/1.73m2) from serum creatinine (mg/dL), age and sex, "
                       "CKD-EPI 2021 equation.",
        "parameters": {"creatinine_mg_dl": "number", "age": "integer", "sex": "'female' or 'male'"},
    },
    "lookup_protocol": {
        "description": "Return the clinic protocol text for a topic.",
        "parameters": {"topic": "one of 'metformin', 'hypoglycaemia'"},
    },
}

def egfr_ckd_epi_2021(creatinine_mg_dl, age, sex):
    k, a = (0.7, -0.241) if sex == "female" else (0.9, -0.302)
    ratio = creatinine_mg_dl / k
    egfr = (142 * min(ratio, 1) ** a * max(ratio, 1) ** -1.200 * 0.9938 ** age
            * (1.012 if sex == "female" else 1))
    return round(egfr)

def lookup_protocol(topic):
    return {"metformin": "Do not start metformin if eGFR is below 45. Stop it if eGFR falls below 30.",
            "hypoglycaemia": "Treat glucose below 70 mg/dL with 15 g fast-acting carbohydrate."}[topic]

print(json.dumps(TOOLS["egfr_ckd_epi_2021"], indent=1))
print(egfr_ckd_epi_2021(1.4, 68, "female"))

# {
#  "description": "Estimate eGFR (mL/min/1.73m2) from serum creatinine (mg/dL), age and sex, CKD-EPI 2021 equation.",
#  "parameters": {
#   "creatinine_mg_dl": "number",
#   "age": "integer",
#   "sex": "'female' or 'male'"
#  }
# }
# 41

### The agent loop

Continuing from the previous block: a minimal agent loop with an allow-list and a step limit.

In [ ]:
def scripted_model(messages):
    """Stand-in for a language model. A real model decides these steps itself;
    here they are fixed so the loop runs without an API."""
    tool_results = [m for m in messages if m["role"] == "tool"]
    if len(tool_results) == 0:
        return {"tool_call": {"name": "egfr_ckd_epi_2021",
                              "arguments": {"creatinine_mg_dl": 1.4, "age": 68, "sex": "female"}}}
    if len(tool_results) == 1:
        return {"tool_call": {"name": "lookup_protocol", "arguments": {"topic": "metformin"}}}
    egfr = tool_results[0]["content"]
    return {"answer": f"Estimated eGFR is {egfr} mL/min/1.73m2. The protocol says not to start metformin "
                      f"below 45, so it should not be started; please confirm with the prescriber. "
                      f"[tools: egfr_ckd_epi_2021, lookup_protocol]"}

IMPLEMENTATIONS = {"egfr_ckd_epi_2021": egfr_ckd_epi_2021, "lookup_protocol": lookup_protocol}

def run_agent(question, max_steps=5):
    messages = [{"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        reply = scripted_model(messages)
        if "answer" in reply:
            return reply["answer"], messages
        call = reply["tool_call"]
        if call["name"] not in IMPLEMENTATIONS:                      # allow-list
            raise ValueError(f"Unknown tool {call['name']}")
        result = IMPLEMENTATIONS[call["name"]](**call["arguments"])
        print(f"step {step}: {call['name']}({call['arguments']}) -> {result}")
        messages.append({"role": "tool", "name": call["name"], "content": result})
    raise RuntimeError("Step limit reached without an answer")

answer, log = run_agent("Synthetic case: 68-year-old woman, creatinine 1.4 mg/dL. Can metformin be started?")
print("\n" + answer)

# step 1: egfr_ckd_epi_2021({'creatinine_mg_dl': 1.4, 'age': 68, 'sex': 'female'}) -> 41
# step 2: lookup_protocol({'topic': 'metformin'}) -> Do not start metformin if eGFR is below 45. Stop it if eGFR falls below 30.
#
# Estimated eGFR is 41 mL/min/1.73m2. The protocol says not to start metformin below 45, so it should not be started; please confirm with the prescriber. [tools: egfr_ckd_epi_2021, lookup_protocol]

### Validating tool inputs

Continuing from the tools block: a validating wrapper around the eGFR tool.

In [ ]:
def validated_egfr(creatinine_mg_dl, age, sex):
    problems = []
    if not 0.2 <= creatinine_mg_dl <= 20:
        problems.append(f"creatinine {creatinine_mg_dl} mg/dL is implausible (was it µmol/L?)")
    if not 18 <= age <= 120:
        problems.append(f"age {age} outside the adult range of this equation")
    if sex not in ("female", "male"):
        problems.append(f"sex must be 'female' or 'male', got {sex!r}")
    if problems:
        return {"error": problems}            # returned to the model, never silently 'fixed'
    return {"egfr": egfr_ckd_epi_2021(creatinine_mg_dl, age, sex)}

print(validated_egfr(1.4, 68, "female"))
print(validated_egfr(124, 68, "female"))     # creatinine given in µmol/L by mistake

# {'egfr': 41}
# {'error': ['creatinine 124 mg/dL is implausible (was it µmol/L?)']}

## Exercise

**Goal:** Build a small tool-using agent loop, add validation and safeguards, and test how it handles bad inputs.

**Data:** Synthetic cases written in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Reproduce the tools and the scripted agent loop.

In [ ]:
answer, log = run_agent("Synthetic case: 68-year-old woman, creatinine 1.4 mg/dL. Can metformin be started?")
print(answer)

### Step 2

Replace the eGFR tool with the validating version and make the scripted model pass creatinine in µmol/L; handle the error by converting units (÷ 88.4) and retrying.

In [ ]:
import datetime

def run_logged_agent(question, model, implementations, max_steps=6):
    """The same loop, but logging every model request and tool result, and returning tool errors to the model."""
    messages = [{"role": "user", "content": question, "time": datetime.datetime.now().isoformat(timespec="seconds")}]
    for step in range(1, max_steps + 1):
        reply = model(messages)
        stamp = datetime.datetime.now().isoformat(timespec="seconds")
        if "answer" in reply:
            messages.append({"role": "assistant", "content": reply["answer"], "time": stamp})
            return reply["answer"], messages
        call = reply["tool_call"]
        messages.append({"role": "assistant", "tool_call": call, "time": stamp})
        if call["name"] not in implementations:                                # allow-list
            result = {"error": [f"tool {call['name']!r} is not allowed"]}
        else:
            result = implementations[call["name"]](**call["arguments"])
        print(f"step {step}: {call['name']}({call['arguments']}) -> {result}")
        messages.append({"role": "tool", "name": call["name"], "content": result, "time": stamp})
    return "Stopped: step limit reached without an answer.", messages

def unit_aware_model(messages):
    """Scripted model: first sends creatinine in µmol/L by mistake, then converts after the tool's error."""
    results = [m for m in messages if m["role"] == "tool"]
    if not results:
        return {"tool_call": {"name": "egfr_ckd_epi_2021",
                              "arguments": {"creatinine_mg_dl": 124, "age": 68, "sex": "female"}}}
    last = results[-1]["content"]
    if isinstance(last, dict) and "error" in last and len(results) == 1:
        mg_dl = round(124 / 88.4, 2)                       # µmol/L -> mg/dL
        return {"tool_call": {"name": "egfr_ckd_epi_2021",
                              "arguments": {"creatinine_mg_dl": mg_dl, "age": 68, "sex": "female"}}}
    if len(results) == 2:
        return {"tool_call": {"name": "lookup_protocol", "arguments": {"topic": "metformin"}}}
    egfr = results[1]["content"]["egfr"]
    return {"answer": f"Creatinine 124 µmol/L = {round(124 / 88.4, 2)} mg/dL; estimated eGFR {egfr} mL/min/1.73m2. "
                      f"The protocol says not to start metformin below 45; please confirm with the prescriber."}

SAFE_TOOLS = {"egfr_ckd_epi_2021": validated_egfr, "lookup_protocol": lookup_protocol}
answer2, log2 = run_logged_agent("Synthetic case: 68-year-old woman, creatinine 124 µmol/L. Can metformin be started?",
                                 unit_aware_model, SAFE_TOOLS)
print("\n" + answer2)

The validating tool refuses the implausible value and says why, the model corrects the unit and retries, and
the log shows all of it. The tool never "fixes" the value silently: guessing that 124 means µmol/L is the
model's (checkable) decision, not hidden code.

### Step 3

Make the scripted model request a tool that does not exist and check that the allow-list stops it.

In [ ]:
def overreaching_model(messages):
    results = [m for m in messages if m["role"] == "tool"]
    if not results:
        return {"tool_call": {"name": "order_lab_test", "arguments": {"test": "HbA1c", "patient": "p1"}}}
    return {"answer": "I could not order the test: that tool is not available. Please order it yourself if needed."}

answer3, log3 = run_logged_agent("Order an HbA1c for this synthetic patient.", overreaching_model, SAFE_TOOLS)
print("\n" + answer3)

The allow-list is enforced by the code that runs tools, not by the model: a tool that is not in
`implementations` cannot be run whatever the model asks. Returning the refusal to the model (rather than
crashing) lets it explain the limit to the user.

### Step 4

Write the full message log to a JSON file and read it as an auditor would.

In [ ]:
from pathlib import Path

log_path = Path("agent_log.json")
log_path.write_text(json.dumps(log2, indent=1, default=str))

print(f"Audit view of {log_path} ({log_path.stat().st_size} bytes)\n")
for i, m in enumerate(json.loads(log_path.read_text()), start=1):
    if m["role"] == "user":
        print(f"{i:>2} {m['time']} USER asked: {m['content']}")
    elif "tool_call" in m:
        print(f"{i:>2} {m['time']} MODEL requested {m['tool_call']['name']} with {m['tool_call']['arguments']}")
    elif m["role"] == "tool":
        print(f"{i:>2} {m['time']} TOOL {m['name']} returned {m['content']}")
    else:
        print(f"{i:>2} {m['time']} MODEL answered: {m['content']}")

An auditor should be able to answer from the log alone: what was asked, which tools were called with which
inputs, what each returned (including errors), and how the answer follows from those results. In a real
deployment the log would also record the user, the model and version, and the patient context, and be stored
where it cannot be edited. Logs contain patient data, so they need the same protection as the record itself.

### Step 5

Optional, with an approved model: replace scripted_model with a real tool-calling model and compare its choices on five synthetic cases.

Optional, with an approved model that supports tool calling, and synthetic cases only. Replace
`scripted_model` with a function that sends `messages` and the `TOOLS` descriptions to the model and converts
its reply into `{"tool_call": ...}` or `{"answer": ...}`; every provider's SDK has an equivalent. Keep the same
loop, allow-list and validating tools.

Five synthetic cases to try: creatinine 1.4 mg/dL (68F); creatinine 124 µmol/L (68F); creatinine 0.8 mg/dL
(45M); a 16-year-old (outside the equation's adult range); and a question the tools cannot answer ("what dose
of semaglutide?"). Compare with the scripted model: does the real model call the tools in a sensible order,
convert units, stop when the tools cannot help, and say "confirm with the prescriber" rather than deciding?

---

Lesson: [https://ramkumarmd.com/medical-ai/agentic-ai/agents-and-tool-use](https://ramkumarmd.com/medical-ai/agentic-ai/agents-and-tool-use). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.